In [1]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Velicia\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [2]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Velicia\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [3]:
%pip install gensim

Note: you may need to restart the kernel to use updated packages.Requirement already satisfied: gensim in c:\users\velicia\appdata\local\programs\python\python310\lib\site-packages (4.4.0)



You should consider upgrading via the 'c:\Users\Velicia\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV
from sklearn.metrics import classification_report
import tensorflow as tf
from tensorflow.keras import Sequential, callbacks, layers, optimizers
from tensorflow.keras.layers import LSTM, Dense, Dropout, Bidirectional
import joblib
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import LabelEncoder, StandardScaler
import re
import nltk
import gensim
from gensim.models import Word2Vec

In [5]:
df = pd.read_csv('CV.csv')
df

,user_id,raw_cv_text,extracted_user_skills,last_role
0,1,One97 Communications Limited \nData Scientist ...,"Object Detection, Market Research, Natural Lan...",Data Scientist Jan 2019 to Till Date
1,2,"Attila László Joó, PhD\nExternal advisor\nPhD ...","Virtual, Design, Independent, Research And Dev...",PhD Civil Engineer
2,3,"VINEET\tMATHRADAS\t\tWay\tNo.\t3023,\tHouse\tN...","Business, Reporting, Revenue, Kingdom, Company...","VINEET\tMATHRADAS\t\tWay\tNo.\t3023,\tHouse\tN..."
3,4,Ákos Pohl\nChief Executive Manager\nMSC in Civ...,"Marital Status, Design, Cleaning, Msc, Optimal...","Civil Engineer, CEOS - Civil Engineering Optim..."
4,5,RESUME \nName: ...,"Medical, Marital Status, Notification, Pneumat...",Post applied for: ...
5,6,\nTamas Futo\nExecutive Civil engineer\nMS...,"Kv, Marital Status, Design, Msc, Optimal Solut...",Executive Civil engineer
6,7,Frederick Chen frederi1@andrew.cmu.edu 4257 Br...,"Unix, Voice, Data, Interpersonal Skills, Cloud...",Frederick Chen frederi1@andrew.cmu.edu 4257 Br...
7,8,RESUME\nAMANDEEP SINGH \nElectrical Engineer ...,"Marital Status, Ensure, Reporting, Flexible, M...",Electrical Engineer
8,9,Curriculum Vitae \nKokula.Bhanuchander \n...,"Work Experience, Knowledge, Progress, Air Hand...",Mechanical Engineer.
9,10,W. A. Sullivan CV 1 Walter A. (Bill) Sullivan ...,"Graduate, Reconnaissance, Analysis, San, Earth...",W. A. Sullivan CV 1 Walter A. (Bill) Sullivan ...


### Data Preprocessing

In [6]:
df['raw_cv_text'] = df['raw_cv_text'].str.lower()
df['extracted_user_skills'] = df['extracted_user_skills'].str.lower()
df['last_role'] = df['last_role'].str.lower()

In [7]:
import string

def remove_punctuation(text):
    punctuation_set = set(string.punctuation)
    text_without_punctuation = ''.join(char for char in text if char not in punctuation_set)
    return text_without_punctuation

df['raw_cv_text'] = df['raw_cv_text'].apply(remove_punctuation)
df['extracted_user_skills'] = df['extracted_user_skills'].apply(remove_punctuation)
df['last_role'] = df['last_role'].apply(remove_punctuation)

In [8]:
df.isnull().sum()

user_id                  0
raw_cv_text              0
extracted_user_skills    0
last_role                0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
from nltk.corpus import stopwords
nltk.download('stopwords')

stop_words = set(stopwords.words('indonesian'))

df['raw_cv_text'] = df['raw_cv_text'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop_words])
)

stop_words = set(stopwords.words('english'))

df['raw_cv_text'] = df['raw_cv_text'].apply(
    lambda x: ' '.join([word for word in x.split() if word not in stop_words])
)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Velicia\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [11]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

df['raw_cv_text'] = df['raw_cv_text'].apply(
    lambda x: ' '.join([lemmatizer.lemmatize(word) for word in x.split()])
)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Velicia\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [14]:
def extract_education(text):
    education_keywords = ['b.tech', 'btech', 'mtech', 'm.tech', 'bachelor', 'master', 'phd', 
                          'degree', 'diploma', 'engineering', 'science', 'arts', 'commerce', 
                          'university', 'college', 'institute', 'iit', 'nit']
    education_info = []
    for keyword in education_keywords:
        if keyword in text:
            words = text.split()
            for i, word in enumerate(words):
                if keyword in word:
                    start = max(0, i-5)
                    end = min(len(words), i+6)
                    context = ' '.join(words[start:end])
                    if context not in education_info:
                        education_info.append(context)
    return ' | '.join(education_info[:3]) if education_info else 'not specified'

def extract_skills(text):
    skill_keywords = ['python', 'java', 'javascript', 'cpp', 'c++', 'sql', 'r programming',
                      'machine learning', 'deep learning', 'tensorflow', 'pytorch', 'keras',
                      'nlp', 'computer vision', 'data science', 'data analysis', 'analytics',
                      'react', 'angular', 'nodejs', 'django', 'flask', 'aws', 'gcp', 'azure',
                      'docker', 'kubernetes', 'git', 'excel', 'tableau', 'power bi', 'spark',
                      'hadoop', 'scala', 'rust', 'golang', 'typescript', 'mongodb', 'postgresql', 'analysis', 'numpy',
                      'pandas', 'matplotlib', 'seaborn', 'scikit-learn', 'xgboost', 'lightgbm', 'natural language',
                      'computer vision', 'reinforcement learning', 'time series', 'big data',
                      'cloud computing', 'devops', 'agile', 'scrum', 'project management'
                      'sotftware', 'data', 'visualization','statistic', 'algorithm', 'model',
                      'customer', 'engineering', 'design']
    
    found_skills = []
    for skill in skill_keywords:
        if skill in text.lower():
            found_skills.append(skill)
    return ', '.join(found_skills) if found_skills else 'not specified'

def extract_projects(text):
    project_keywords = ['project', 'built', 'developed', 'created', 'implemented', 'application',
                       'system', 'platform', 'tool', 'framework', 'model']
    projects = []
    words = text.split()
    
    for i, word in enumerate(words):
        if any(keyword in word.lower() for keyword in project_keywords):
            start = max(0, i-2)
            end = min(len(words), i+8)
            context = ' '.join(words[start:end])
            if context not in projects and len(context) > 10:
                projects.append(context)
    
    return ' | '.join(projects[:3]) if projects else 'not specified'

def extract_experience(text):
    role_keywords = ['manager', 'engineer', 'scientist', 'developer', 'analyst', 'architect',
                    'consultant', 'lead', 'senior', 'junior', 'intern', 'associate', 'officer',
                    'specialist', 'coordinator', 'director', 'head']
    
    experiences = []
    words = text.split()
    
    for i, word in enumerate(words):
        if any(keyword in word.lower() for keyword in role_keywords):
            start = max(0, i-2)
            end = min(len(words), i+5)
            context = ' '.join(words[start:end])
            if context not in experiences:
                experiences.append(context)
    
    return ' | '.join(experiences[:3]) if experiences else 'not specified'

def extract_major(text):
    majors = ['computer science', 'engineering', 'information technology', 'civil engineering',
             'mechanical engineering', 'electrical engineering', 'software engineering',
             'data science', 'artificial intelligence', 'business', 'finance', 'economics',
             'psychology', 'mathematics', 'physics', 'chemistry']
    
    found_majors = []
    for major in majors:
        if major in text.lower():
            found_majors.append(major)
    
    return ', '.join(found_majors) if found_majors else 'other'

df['education'] = df['raw_cv_text'].apply(extract_education)
df['major'] = df['raw_cv_text'].apply(extract_major)
df['extracted_skills_structured'] = df['raw_cv_text'].apply(extract_skills)
df['projects'] = df['raw_cv_text'].apply(extract_projects)
df['experience'] = df['raw_cv_text'].apply(extract_experience)
df

,user_id,raw_cv_text,extracted_user_skills,last_role,education,major,extracted_skills_structured,projects,experience
0,1,one97 communication limited data scientist jan...,object detection market research natural langu...,data scientist jan 2019 to till date,probability estimate premium amount charged bt...,"computer science, engineering, data science","python, java, machine learning, deep learning,...",expand arsenal application building work diffe...,limited data scientist jan 2019 till date | co...
1,2,attila lászló joó phd external advisor phd civ...,virtual design independent research and develo...,phd civil engineer,attila lászló joó phd external advisor phd civ...,"engineering, civil engineering","analysis, model, engineering, design",bridge numerical modelling steel joint virtual...,phd civil engineer present workplace assistant...
2,3,vineet mathradas way 3023 house 1918 po box 60...,business reporting revenue kingdom company mai...,vineet\tmathradas\t\tway\tno\t3023\thouse\tno\...,2017 london school economics political science...,"business, finance, economics","analysis, model",comprehensive financial model comprising annua...,class india senior school examination grade 12...
3,4,ákos pohl chief executive manager msc civil en...,marital status design cleaning msc optimal sol...,civil engineer ceos civil engineering optimal...,email apohlce oseu qualification msc diploma c...,"engineering, civil engineering, economics","excel, engineering, design",isd dunaferr project 2008 hexagon team award 2...,chief executive manager msc civil engineering ...
4,5,resume name rama krushna dash post applied mec...,medical marital status notification pneumatic ...,post applied for ...,berhampur odisha 71 recently completed btech ...,"engineering, mechanical engineering","engineering, design",located centrally project site ii preventive m...,applied mechanical engineer contact 9078684961...
5,6,tamas futo executive civil engineer msc civil ...,kv marital status design msc optimal solutions...,executive civil engineer,0427 email tfutoceoseu qualification msc diplo...,"engineering, civil engineering, economics","excel, analysis, engineering, design",small building system –research report departm...,executive civil engineer msc civil engineering...
6,7,frederick chen frederi1andrewcmuedu 4257 bryn ...,unix voice data interpersonal skills cloud ada...,frederick chen frederi1andrewcmuedu 4257 bryn ...,university pittsburgh pa gpa 308 bachelor scie...,"computer science, engineering, electrical engi...","java, javascript, data, algorithm, model, engi...",bigdata business model • integrated cloud serv...,program allows junior senior take college cour...
7,8,resume amandeep singh electrical engineer dob ...,marital status ensure reporting flexible maint...,electrical engineer,team work integrity educational profile degree...,engineering,"engineering, design",real life application  person cooperative sup...,singh electrical engineer dob 09feb1986 nation...
8,9,curriculum vitae kokulabhanuchander mechanical...,work experience knowledge progress air handlin...,mechanical engineer,2hvac education group branch duration 1btech m...,"engineering, mechanical engineering","engineering, design",handling mechanical project hvac engineer wid...,kokulabhanuchander mechanical engineer mail id...
9,10,w sullivan cv 1 walter bill sullivan departmen...,graduate reconnaissance analysis san earth sci...,w a sullivan cv 1 walter a bill sullivan depar...,evolution western north america education phd ...,information technology,"rust, analysis, model, design",strikeslip fault system case study kellyland f...,held 2014–present associate professor geology ...


In [ ]:
from nltk.tokenize import word_tokenize
nltk.download('punkt')

tokenized_data = [word_tokenize(text) for text in df['raw_cv_text']]
model = Word2Vec(sentences=tokenized_data, vector_size=100, window=5, min_count=1, workers=4)

In [16]:
df.to_csv('cv_processed.csv')

### Pipeline

### Model